# SPACE Headless API Example

Demonstrates the `space_api.run_pipeline()` headless API for the SPACE
(Sequence Protein Alignment and Conservation Engine) package.

Interpreter: `/home/dzyla/miniconda3/envs/space/bin/python`


In [ ]:
import space_api as sa


## Run the full pipeline headlessly

`space_api.run_pipeline()` installs the headless Streamlit stub internally
(`headless_streamlit.install()`), so the entire `space.*` pipeline runs without
a browser. The NCBI email is `dzyla@lji.org`.


In [ ]:
result = sa.run_pipeline(
    query="Hendra henipavirus F",
    email="dzyla@lji.org",
    data_source="UniProt",
    max_seqs=50,
    out_dir="runs/hendra_f",
)

print("Pipeline completed. Output dir:", result["out_dir"])
print("FASTA:", result["fasta_path"])
print("MSA:", result["msa_outfile"])
print("ALN:", result["aln_file"])
print("Tree:", result["tree_file"])


## Inspect the outputs

The returned dict contains: `al2co_df`, `conservation_df`, `mutations_df`,
`tree_file`, and optionally `pdb_result` (when `map_to_structure=True`).


In [ ]:
# al2co conservation scores (per-residue)
print("=== al2co_df ===")
print(result["al2co_df"].head())

# point mutations
print("\n=== mutations_df ===")
print(result["mutations_df"].head())

# conservation summary
print("\n=== conservation_df ===")
print(result["conservation_df"].head())


## Optional structural mapping (Step 7)

Map al2co scores onto a 3D structure by setting `map_to_structure=True` and
providing a `uniprot_id` (fetches the AlphaFold PDB) or `own_pdb`.


In [ ]:
result_struct = sa.run_pipeline(
    query="Hendra henipavirus F",
    email="dzyla@lji.org",
    data_source="UniProt",
    max_seqs=50,
    out_dir="runs/hendra_f_struct",
    map_to_structure=True,
    uniprot_id="Q8IHW4",
)

print("Structural mapping result:", result_struct["pdb_result"])


## Pitfall: identical sequences yield zero variance

If all aligned sequences are identical, every column has zero variance and
al2co returns NaN conservation scores. Deduplicate identical sequences before
running the pipeline.


In [ ]:
import tempfile, os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

identical = [
    SeqRecord(Seq("ACDEFGHIKLMNPQRSTVWY"), id="seq1"),
    SeqRecord(Seq("ACDEFGHIKLMNPQRSTVWY"), id="seq2"),
    SeqRecord(Seq("ACDEFGHIKLMNPQRSTVWY"), id="seq3"),
]
with tempfile.TemporaryDirectory() as tmpdir:
    fasta_path = os.path.join(tmpdir, "identical.fasta")
    SeqIO.write(identical, fasta_path, "fasta")
    result_id = sa.run_pipeline(
        fasta_path=fasta_path,
        out_dir=os.path.join(tmpdir, "identical_run"),
    )
    print("al2co scores for identical sequences (all zero variance -> NaN):")
    print(result_id["al2co_df"].head())
    print("Any NaN?", result_id["al2co_df"].isna().any().any())


## Pipeline stage reference

| Stage | Function | Output |
|---|---|---|
| Step 0 - fetch | `fetch_sequences` | `sequences.fasta` |
| Step 1 - pairwise align | `perform_alignment` | identity/coverage, mapping |
| Step 2 - filter | `filter_sequences` | filtered sequences |
| Step 3 - MSA | `perform_msa_pyfamsa` | `msa_out.fasta`, `.aln` |
| Step 4 - al2co | `run_al2co` | `al2co_df` |
| Step 5 - mutations | `parse_mutations` | `mutations_df` |
| Step 6 - tree | `generate_phylogenetic_tree` | tree file (if <=200 seqs) |
| Step 7 - structure | `map_structure` | `pdb_result` (optional) |
